# 02 commit lint

Question: does the commit lint of Golem, `.github/scripts/commit_lint.py`, pass the history of this
revision and refuse every break of its eight rules, and does Commitizen, configured by `.cz.toml`,
agree with it on the subject and the layout of a message.

CONTRIBUTING.md section 3 states the form: a Conventional Commit as Commitizen writes it, with a
body and trailers in the form of the Linux kernel. The notebook checks the history of the
repository first, then a good message and one break of each rule, as a message file and as a commit
in a scratch repository. Commitizen judges only the subject and the empty line after it, so a case
names what `cz check` must say, or nothing where the rule is one of the kernel alone.

In [ ]:
# Papermill overrides this cell.
# The tree under test, the root of the repository.
golem_root = "."

# The sign off every commit of Golem ends with.
sign_off = "Signed-off-by: Florian Kleber <opensource@skytekx.com>"

## Setup

The cell finds the repository and prints the versions of Python, git and Commitizen.

In [ ]:
import shutil
import sys
import tempfile
from pathlib import Path

for _base in [Path.cwd(), *Path.cwd().parents]:
    if (_base / "tests" / "harness" / "__init__.py").is_file():
        sys.path.insert(0, str(_base / "tests"))
        break

from harness import paths, run

P = paths(golem_root)
LINT = P.root / ".github" / "scripts" / "commit_lint.py"
CZ_CONFIG = P.root / ".cz.toml"
CZ = [sys.executable, "-m", "commitizen", "--config", str(CZ_CONFIG)]
SCRATCH = Path(tempfile.mkdtemp(prefix="golem-02-"))

print("python:", sys.version.split()[0])
print("git:", run(["git", "--version"], check=True).stdout.strip())
print("commitizen:", run(CZ + ["version"], check=True).stdout.strip())
print("lint:", P.rel(LINT))

## The history of this revision

Every commit of the branch must pass both tools, so that CI, which runs the same commands, is green.
A shallow clone holds only part of the history, so the cell checks what the clone holds.

In [ ]:
lint = run(["python3", "-I", LINT, "--range", "HEAD", "--root", P.root])
print(lint.stdout.strip().splitlines()[-1])
assert lint.returncode == 0, "the commit lint fails the history:\n" + lint.stdout
cz = run(CZ + ["check", "--rev-range", "HEAD"], cwd=P.root)
print(cz.stdout.strip() or cz.stderr.strip())
assert cz.returncode == 0, "cz check fails the history:\n" + cz.stdout + cz.stderr

## The cases

Each case is a name, a message, the rule the lint must report or `None` for a pass, and what
`cz check` must say: `True` for pass, `False` for fail, `None` where Commitizen does not judge the
rule. The lint reads the cases in a small repository of their own with the one commit
`feat(essentials): add pay`, so a Fixes trailer names a real hash and its subject.

In [ ]:
FIXTURE = SCRATCH / "fixture"
run(["git", "init", "-q", "-b", "main", FIXTURE], check=True)
run(["git", "-c", "user.name=Golem Harness", "-c", "user.email=harness@example.org", "-c", "commit.gpgsign=false",
     "-C", FIXTURE, "commit", "-q", "--allow-empty", "-m", "feat(essentials): add pay"], check=True)
FIXED = run(["git", "-C", FIXTURE, "rev-parse", "HEAD"], check=True).stdout.strip()[:12]

BODY = (
    "The pay command must never move money from the receiver to the sender.\n"
    "Golem parses every amount as whole cents and refuses anything that is\n"
    "not above zero before it touches an account."
)
SUBJECT = "feat(essentials): refuse amounts below one cent in pay"


def message(subject=SUBJECT, body=BODY, trailers=None):
    trailers = [sign_off] if trailers is None else trailers
    parts = [subject, body] + (["\n".join(trailers)] if trailers else [])
    return "\n\n".join(part for part in parts if part is not None) + "\n"


CASES = [
    ("a good message", message(), None, True),
    ("a good message with every kind of trailer", message(trailers=[
        "Fixes: %s (\"feat(essentials): add pay\")" % FIXED,
        "Closes: https://github.com/skytekx/golem/issues/12",
        "Link: https://github.com/skytekx/golem/pull/13",
        "Reported-by: A Player <player@example.org>",
        "Co-Authored-By: Claude Opus 5.5 <noreply@anthropic.com>",
        sign_off]), None, True),
    ("a co-author as GitHub spells it", message(trailers=["Co-authored-by: A Reviewer <reviewer@example.org>", sign_off]),
     None, True),
    ("a summary of one character", message(subject="fix(essentials): a"), None, True),
    ("a breaking change", message(subject="feat(api)!: drop the field uptime of Server"), None, True),
    ("a revert", message(subject="revert(essentials): refuse amounts below one cent in pay"), None, True),
    ("a long URL in the body", message(body=BODY + "\nhttps://example.org/" + "a" * 80), None, True),
    ("an indented log line in the body", message(body=BODY + "\n\n    [12:00:00] [Server thread/INFO]: " + "x" * 60), None, True),
    ("1: no scope", message(subject="feat: refuse amounts below one cent in pay"), 1, False),
    ("1: a type outside the list", message(subject="style(essentials): refuse amounts in pay"), 1, False),
    ("1: the subject of the old form", message(subject="essentials: refuse amounts below one cent"), 1, False),
    ("1: a summary in upper case", message(subject="feat(essentials): Refuse amounts below one cent"), 1, False),
    ("1: a summary with a full stop", message(subject="feat(essentials): refuse amounts below one cent."), 1, False),
    ("1: a fixup of git commit --fixup", message(subject="fixup! feat(essentials): refuse amounts below one cent"), 1, False),
    ("1: a squash of git commit --squash", message(subject="squash! feat(essentials): refuse amounts below one cent"), 1, False),
    ("1: a revert of git revert", message(subject='Revert "feat(essentials): refuse amounts below one cent"'), 1, False),
    ("2: a subject of 73 characters", message(subject="feat(essentials): " + "a" * 55), 2, False),
    ("3: no empty line after the subject", SUBJECT + "\n" + BODY + "\n\n" + sign_off + "\n", 3, False),
    ("3: no body", message(body=None), 3, None),
    ("4: a body line of 73 columns", message(body=BODY + "\n" + "word " * 14 + "abc"), 4, True),
    ("4: a wide body line shaped as a trailer", message(body=BODY + "\nProblem: " + ("word " * 19).rstrip()), 4, True),
    ("a wide trailer in the last paragraph", message(trailers=[
        "Link: https://github.com/skytekx/golem/pull/13#discussion_r" + "1" * 40,
        "Reported-by: A Player Of The Live Server Of SkyTekx <a.player.of.the.live.server@example.org>",
        sign_off]), None, True),
    ("5: no trailers", message(trailers=[]), 5, True),
    ("5: a trailer of an unknown kind", message(trailers=["Approved-by: Someone <someone@example.org>", sign_off]), 5, True),
    ("5: a trailer spelled in lower case", message(trailers=["signed-off-by: Florian Kleber <opensource@skytekx.com>"]), 5, True),
    ("5: prose in the block of trailers", message(trailers=["And one more thing.", sign_off]), 5, True),
    ("5: trailers apart from the last block", message(body=BODY + "\n\nFixes: %s (\"feat(essentials): add pay\")" % FIXED), 5, True),
    ("6: no Signed-off-by", message(trailers=["Closes: https://github.com/skytekx/golem/issues/12"]), 6, True),
    ("6: Signed-off-by not last", message(trailers=[sign_off, "Closes: https://github.com/skytekx/golem/issues/12"]), 6, True),
    ("6: Signed-off-by without an address", message(trailers=["Signed-off-by: Florian Kleber"]), 6, True),
    ("7: Fixes with a short hash", message(trailers=["Fixes: %s (\"feat(essentials): add pay\")" % FIXED[:7], sign_off]), 7, True),
    ("7: Fixes without the subject", message(trailers=["Fixes: " + FIXED, sign_off]), 7, True),
    ("7: Fixes naming no commit", message(trailers=["Fixes: 0123456789ab (\"feat(essentials): add pay\")", sign_off]), 7, True),
    ("7: Fixes with another subject", message(trailers=["Fixes: %s (\"feat(essentials): add the warp list\")" % FIXED, sign_off]), 7, True),
    ("7: Closes with an issue number", message(trailers=["Closes: #12", sign_off]), 7, True),
    ("8: the same trailer twice", message(trailers=[sign_off, sign_off]), 8, True),
]
print("cases:", len(CASES))

In [ ]:
rows, failures = [], []
for index, (name, text, rule, cz_expected) in enumerate(CASES):
    path = SCRATCH / ("case-%02d.txt" % index)
    path.write_text(text, encoding="utf-8")
    lint = run(["python3", "-I", LINT, "--message-file", path, "--root", FIXTURE])
    if rule is None:
        lint_ok = lint.returncode == 0 and not lint.stdout.strip()
    else:
        lint_ok = lint.returncode == 1 and ": rule %d: " % rule in lint.stdout
    cz = run(CZ + ["check", "--commit-msg-file", path], cwd=P.root)
    cz_ok = cz_expected is None or (cz.returncode == 0) == cz_expected
    rows.append((name, "pass" if rule is None else "rule %d" % rule, "ok" if lint_ok else "WRONG",
                 {True: "pass", False: "fail", None: "-"}[cz_expected], "pass" if cz.returncode == 0 else "fail", "ok" if cz_ok else "WRONG"))
    if not (lint_ok and cz_ok):
        failures.append((name, text, lint.stdout, cz.stdout + cz.stderr))

width = max(len(row[0]) for row in rows)
print("%-*s  %-7s %-6s  %-6s %-6s %s" % (width, "case", "lint", "", "cz", "got", ""))
for row in rows:
    print("%-*s  %-7s %-6s  %-6s %-6s %s" % (width, *row))
for name, text, lint_output, cz_output in failures:
    print("\n--- %s\n%s\n--- lint\n%s--- cz\n%s" % (name, text, lint_output, cz_output))
assert not failures, "%d of %d cases went wrong" % (len(failures), len(CASES))

## Commits in a repository

The lint reads a range of commits as CI does: by default the commits of HEAD that `origin/main`
does not hold, or all of HEAD while `origin/main` does not exist. A merge commit is left out, since
git writes its message. The cell makes a scratch repository with a good commit, a merge and a bad
commit, and checks that the range finds the bad one alone.

In [ ]:
repository = SCRATCH / "repository"
identity = ["-c", "user.name=Golem Harness", "-c", "user.email=harness@example.org", "-c", "commit.gpgsign=false"]


def git(*args):
    return run(["git", *identity, "-C", repository, *args], check=True)


run(["git", "init", "-q", "-b", "main", repository], check=True)
(repository / "a.txt").write_text("a\n")
git("add", "a.txt")
git("commit", "-q", "-m", message().rstrip("\n"))
git("checkout", "-q", "-b", "side")
(repository / "b.txt").write_text("b\n")
git("add", "b.txt")
git("commit", "-q", "-m", message(subject="feat(essentials): add the warp list").rstrip("\n"))
git("checkout", "-q", "main")
git("merge", "-q", "--no-ff", "--no-edit", "side")
(repository / "c.txt").write_text("c\n")
git("add", "c.txt")
git("commit", "-q", "-m", "essentials: fix it.")

whole = run(["python3", "-I", LINT, "--root", repository])
print(whole.stdout.strip())
assert whole.returncode == 1 and "range HEAD, commits 3, failed 1" in whole.stdout, whole.stdout
assert "essentials: fix it.: rule 1:" in whole.stdout, whole.stdout

git("update-ref", "refs/remotes/origin/main", "HEAD~1")
ahead = run(["python3", "-I", LINT, "--root", repository])
print(ahead.stdout.strip().splitlines()[-1])
assert ahead.returncode == 1 and "range origin/main..HEAD, commits 1, failed 1" in ahead.stdout, ahead.stdout

git("update-ref", "refs/remotes/origin/main", "HEAD")
clean = run(["python3", "-I", LINT, "--root", repository])
print(clean.stdout.strip())
assert clean.returncode == 0 and "commits 0, failed 0" in clean.stdout, clean.stdout

## A line that opens with #

Git keeps a line that opens with `#` in a message given with `-m` or `-F`, so a commit of a range
holds it and the lint judges it like every other line. The cell commits a subject followed by
`#12` and a body line of 104 columns that opens with `#`, which break rules 3 and 4, and a body
whose only paragraph opens with `#12`, which passes.

In [ ]:
hashed = SCRATCH / "hashed"
run(["git", "init", "-q", "-b", "main", hashed], check=True)
run(["git", *identity, "-C", hashed, "commit", "-q", "--allow-empty", "-m", message().rstrip("\n")], check=True)


def committed(*paragraphs):
    # A commit whose message is `paragraphs`, as git commit -m writes them, judged as a range.
    arguments = [part for paragraph in paragraphs for part in ("-m", paragraph)]
    run(["git", *identity, "-C", hashed, "commit", "-q", "--allow-empty", *arguments], check=True)
    stored = run(["git", "-C", hashed, "log", "-1", "--format=%B"], check=True).stdout
    return stored, run(["python3", "-I", LINT, "--root", hashed, "--range", "HEAD~1..HEAD"])


wide = "# " + "x" * 102
stored, judged = committed("feat(world): add the home list\n#12", BODY + "\n" + wide, sign_off)
print(judged.stdout.strip())
assert "\n#12\n" in stored and wide in stored, "git dropped the lines that open with #:\n" + stored
assert judged.returncode == 1 and "rule 3: no empty line follows the subject" in judged.stdout, judged.stdout
assert "rule 4: line 7 is 104 columns wide" in judged.stdout, judged.stdout
stored, judged = committed("fix(essentials): keep the issue number", "#12 names the issue that this change solves.", sign_off)
print(judged.stdout.strip().splitlines()[-1])
assert "\n#12 names" in stored and judged.returncode == 0, judged.stdout

## The hook

`tools/hooks/commit-msg` runs the lint on the message of a `git commit` before git writes it, so a
bad message never becomes a commit that way. The cell installs the hooks of the repository in the
scratch repository and checks that git refuses a bad message and takes a good one. The cells after
it hold the hook to an editor, to a merge and to `git revert`, which commits without it.

In [ ]:
hooks = repository / "hooks"
shutil.copytree(P.root / "tools" / "hooks", hooks)
(hooks / "post-commit").unlink()
shutil.copytree(P.root / ".github", repository / ".github")
git("config", "core.hooksPath", "hooks")
(repository / "d.txt").write_text("d\n")
git("add", "d.txt")
refused = run(["git", *identity, "-C", repository, "commit", "-q", "-m", "fix: Bad."])
# Git prints the output of a hook on its standard error.
said = refused.stdout + refused.stderr
print("bad message:", refused.returncode, said.strip().splitlines()[0] if said.strip() else "")
assert refused.returncode != 0 and "rule 1" in said, said
taken = run(["git", *identity, "-C", repository, "commit", "-q", "-m", message(subject="feat(world): add the merge radius").rstrip("\n")])
assert taken.returncode == 0, taken.stdout + taken.stderr
print("good message:", run(["git", "-C", repository, "log", "-1", "--format=%s"], check=True).stdout.strip())

The hook reads the file as git will store it. Git cuts an edited message at the cut line of
`git commit --verbose`, which its diff follows, and drops the lines that open with the comment
character of `core.commentChar`, but it keeps them when no editor ran, which it tells the hook
with `GIT_EDITOR` set to `:`. The cell commits a good message through an editor with `--verbose`,
with `commit.verbose`, with `core.commentChar` set to `;` and set to `auto` above a body line that
opens with `#`, and with `-m` and such a body line. Git must take each one, and still refuse a bad
message written in the editor.

In [ ]:
import os
import shlex
import subprocess

EDITOR = SCRATCH / "editor.py"
EDITOR.write_text(
    "import sys\n"
    "from pathlib import Path\n"
    "# Writes the text of editor.txt above what git put into the file.\n"
    "path = Path(sys.argv[1])\n"
    "path.write_text(Path(sys.argv[0]).with_suffix('.txt').read_text() + path.read_text())\n"
)


def edited(text, *arguments):
    # A commit through an editor that writes `text` above the file of git, with the hook installed.
    (SCRATCH / "editor.txt").write_text(text)
    with open(repository / "e.txt", "a") as stream:
        stream.write(" ".join(arguments) + "\n")
    git("add", "e.txt")
    editor = "%s %s" % (shlex.quote(sys.executable), shlex.quote(str(EDITOR)))
    made = subprocess.run(["git", *identity, "-C", str(repository), *arguments], capture_output=True, text=True,
                          env=dict(os.environ, GIT_EDITOR=editor))
    written = (repository / ".git" / "COMMIT_EDITMSG").read_text()
    return made, written


good = message(subject="feat(world): add the spawn radius")
hashed_body = ["-m", "fix(essentials): keep the issue number", "-m", "#12 names the issue that this change solves.",
               "-m", sign_off]
for label, text, arguments, mark in (
        ("--verbose", good, ["commit", "-v"], "# ------------------------ >8"),
        ("commit.verbose", good, ["-c", "commit.verbose=true", "commit"], "# ------------------------ >8"),
        ("core.commentChar ;", good, ["-c", "core.commentChar=;", "commit"], "; Please enter"),
        ("core.commentChar auto", "", ["-c", "core.commentChar=auto", "commit", "-v", "-e", *hashed_body],
         "; ------------------------ >8")):
    made, written = edited(text, *arguments)
    kept = run(["git", "-C", repository, "log", "-1", "--format=%B"], check=True).stdout
    print("%s: exit %d, the file held %d lines, git stored %d" % (label, made.returncode, len(written.splitlines()),
                                                                 len(kept.splitlines())))
    assert mark in written, "git wrote no %r into the file for %s:\n%s" % (mark, label, written)
    assert made.returncode == 0, "the hook refused a good message with %s:\n%s" % (label, made.stdout + made.stderr)
    assert kept.rstrip("\n") == (text or "\n\n".join(hashed_body[1::2])).rstrip("\n"), kept

(repository / "f.txt").write_text("f\n")
git("add", "f.txt")
plain = run(["git", *identity, "-C", repository, "commit", "-q", *hashed_body])
print("-m with a line that opens with #: exit", plain.returncode)
assert plain.returncode == 0, plain.stdout + plain.stderr
assert "\n#12 names" in run(["git", "-C", repository, "log", "-1", "--format=%B"], check=True).stdout
made, written = edited("fix: Bad.\n", "commit", "-v")
print("a bad message with --verbose: exit", made.returncode)
assert made.returncode != 0 and "rule 1" in made.stderr, made.stdout + made.stderr

Git runs the hook for a merge as well, on the message that it writes itself. A range leaves out
every merge, and so does the hook while git makes one: `git merge` hands it `MERGE_MSG`, and a
`git commit` that concludes a merge runs while `MERGE_HEAD` exists. The cell merges a branch with
the hook installed, once in one step and once with `--no-commit` and a `git commit` after it.

In [ ]:
git("reset", "-q")
for branch, steps in (("hooked-side", [["merge", "--no-ff", "--no-edit", "hooked-side"]]),
                      ("hooked-other", [["merge", "--no-ff", "--no-commit", "hooked-other"], ["commit", "--no-edit"]])):
    git("checkout", "-q", "-b", branch)
    (repository / (branch + ".txt")).write_text(branch + "\n")
    git("add", branch + ".txt")
    git("commit", "-q", "-m", message(subject="feat(world): add the file %s" % branch).rstrip("\n"))
    git("checkout", "-q", "main")
    for step in steps:
        done = run(["git", *identity, "-C", repository, *step])
        print("git %s: exit %d" % (" ".join(step), done.returncode))
        assert done.returncode == 0, "the hook refused a merge:\n" + done.stdout + done.stderr
    assert run(["git", "-C", repository, "rev-parse", "-q", "--verify", "MERGE_HEAD"]).returncode != 0
    parents = run(["git", "-C", repository, "log", "-1", "--format=%P"], check=True).stdout.split()
    assert len(parents) == 2, "git made no merge of %s" % branch

Git runs no commit-msg hook for `git revert` and `git cherry-pick`, with `--edit` neither, so a
revert of git commits its own subject `Revert "..."`, which only the lint of CI refuses later. A
revert written with `git revert --no-commit` and a `git commit` goes through the hook, which
refuses the subject of git and takes a `revert(scope): summary`. The cell shows both ways.

In [ ]:
(repository / "h.txt").write_text("h\n")
git("add", "h.txt")
git("commit", "-q", "-m", message(subject="feat(world): add the file h").rstrip("\n"))
bypassed = run(["git", *identity, "-C", repository, "revert", "--no-edit", "HEAD"])
subject = run(["git", "-C", repository, "log", "-1", "--format=%s"], check=True).stdout.strip()
print("git revert: exit %d, committed %s" % (bypassed.returncode, subject))
assert bypassed.returncode == 0 and subject == 'Revert "feat(world): add the file h"', bypassed.stderr + subject
later = run(["python3", "-I", LINT, "--root", repository, "--range", "HEAD~1..HEAD"])
assert later.returncode == 1 and ": rule 1: " in later.stdout, later.stdout
git("reset", "-q", "--hard", "HEAD~1")
git("revert", "--no-commit", "HEAD")
refused = run(["git", *identity, "-C", repository, "commit", "-q", "--no-edit"])
print("git revert --no-commit and git commit with the subject of git: exit", refused.returncode)
assert refused.returncode != 0 and ": rule 1: " in refused.stderr, refused.stdout + refused.stderr
written = run(["git", *identity, "-C", repository, "commit", "-q", "-m",
               message(subject="revert(world): drop the file h").rstrip("\n")])
print("git revert --no-commit and git commit with revert(world): exit", written.returncode)
assert written.returncode == 0, written.stdout + written.stderr

## The push after a commit

`tools/hooks/post-commit` pushes the branch after every commit. Only a push that fails on the
network is retried, after pauses of 2, 4, 8 and 16 seconds. A push that origin refuses, such as the
one after an amend of a pushed commit, and a repository without the remote `origin` are reported
at once in one line, since no retry mends them. The cell commits in a clone of a bare repository
with the hook installed and times each commit.

In [ ]:
import socket
import time

pushed = SCRATCH / "pushed"
run(["git", "init", "-q", "--bare", pushed / "origin.git"], check=True)
clone = pushed / "clone"
run(["git", "init", "-q", "-b", "work", clone], check=True)
run(["git", "-C", clone, "remote", "add", "origin", pushed / "origin.git"], check=True)
(clone / "hooks").mkdir()
shutil.copy2(P.root / "tools" / "hooks" / "post-commit", clone / "hooks" / "post-commit")
run(["git", "-C", clone, "config", "core.hooksPath", "hooks"], check=True)


def timed(*arguments):
    # A commit in the clone with the hook, its seconds and what the hook printed.
    started = time.monotonic()
    made = run(["git", *identity, "-C", clone, "commit", "-q", "--allow-empty", *arguments], timeout=120)
    seconds = time.monotonic() - started
    assert made.returncode == 0, made.stdout + made.stderr
    return seconds, made.stderr


def at_origin():
    return run(["git", "-C", pushed / "origin.git", "rev-parse", "work"], check=True).stdout.strip()


seconds, said = timed("-m", "one")
head = run(["git", "-C", clone, "rev-parse", "HEAD"], check=True).stdout.strip()
print("first commit: %.1f s, origin holds it: %s" % (seconds, at_origin() == head))
assert said == "" and at_origin() == head, said
before = at_origin()
seconds, said = timed("--amend", "-m", "two")
print("amend of the pushed commit: %.1f s, %s" % (seconds, said.strip().splitlines()[-1]))
assert seconds < 10 and said.count("[rejected]") == 1 and "post-commit: origin refused work" in said, said
assert "--force-with-lease" in said and at_origin() == before, said

free = socket.socket()
free.bind(("127.0.0.1", 0))
port = free.getsockname()[1]
free.close()
run(["git", "-C", clone, "remote", "set-url", "origin", "git://127.0.0.1:%d/origin.git" % port], check=True)
seconds, said = timed("-m", "three")
print("origin unreachable: %.1f s, %s" % (seconds, said.strip().splitlines()[-1]))
assert seconds >= 30 and said.count("the network failed the push of work") == 4, said
assert "failed five times" in said, said

run(["git", "-C", clone, "remote", "remove", "origin"], check=True)
seconds, said = timed("-m", "four")
print("no remote origin: %.1f s, %s" % (seconds, said.strip()))
assert seconds < 10 and said.strip() == "post-commit: the repository has no remote origin, so work is not pushed", said

## A shallow clone

The job of the notebooks in CI checks out a shallow clone, which holds HEAD alone. A Fixes trailer
there may name a commit that the clone lacks, so the lint lets a hash pass that a shallow clone
does not hold, and still refuses it in the full repository.

The checkout of a pull request is GitHub's merge commit, `Merge <sha> into <sha>`, and in a shallow
clone it has lost its parents, so `--no-merges` keeps it. The lint leaves out a commit at the edge
of the clone whose message git writes for a merge, and still judges every other commit there.

In [ ]:
deep = SCRATCH / "deep"
run(["git", "init", "-q", "-b", "main", deep], check=True)
run(["git", *identity, "-C", deep, "commit", "-q", "--allow-empty", "-m", message(subject="feat(essentials): add pay").rstrip("\n")], check=True)
first = run(["git", "-C", deep, "rev-parse", "HEAD"], check=True).stdout.strip()[:12]
fixing = message(subject="fix(essentials): refuse amounts below one cent in pay",
                 trailers=["Fixes: %s (\"feat(essentials): add pay\")" % first, sign_off])
run(["git", *identity, "-C", deep, "commit", "-q", "--allow-empty", "-m", fixing.rstrip("\n")], check=True)
shallow = SCRATCH / "shallow"
run(["git", "clone", "-q", "--depth", "1", "file://%s" % deep, shallow], check=True)
assert run(["git", "-C", shallow, "rev-parse", "--is-shallow-repository"], check=True).stdout.strip() == "true"

both = {where: run(["python3", "-I", LINT, "--root", where, "--range", "HEAD"]) for where in (deep, shallow)}
for where, result in both.items():
    print(where.name, result.stdout.strip().splitlines()[-1])
    assert result.returncode == 0, result.stdout
missing = SCRATCH / "missing.txt"
missing.write_text(message(trailers=["Fixes: 0123456789ab (\"feat(essentials): add pay\")", sign_off]))
in_shallow = run(["python3", "-I", LINT, "--message-file", missing, "--root", shallow])
in_deep = run(["python3", "-I", LINT, "--message-file", missing, "--root", deep])
assert in_shallow.returncode == 0, in_shallow.stdout
assert in_deep.returncode == 1 and "rule 7: Fixes names 0123456789ab, which is no commit" in in_deep.stdout, in_deep.stdout
print("a hash the shallow clone lacks passes there and fails in the full repository")

merged = SCRATCH / "merged"
run(["git", "init", "-q", "-b", "main", merged], check=True)
run(["git", *identity, "-C", merged, "commit", "-q", "--allow-empty", "-m", message().rstrip("\n")], check=True)
run(["git", "-C", merged, "checkout", "-q", "-b", "side"], check=True)
run(["git", *identity, "-C", merged, "commit", "-q", "--allow-empty", "-m", message(subject="feat(essentials): add the warp list").rstrip("\n")], check=True)
run(["git", "-C", merged, "checkout", "-q", "main"], check=True)
run(["git", *identity, "-C", merged, "commit", "-q", "--allow-empty", "-m", message(subject="feat(essentials): add the home list").rstrip("\n")], check=True)
side = run(["git", "-C", merged, "rev-parse", "side"], check=True).stdout.strip()
main = run(["git", "-C", merged, "rev-parse", "main"], check=True).stdout.strip()
run(["git", *identity, "-C", merged, "merge", "-q", "--no-ff", "-m", "Merge %s into %s" % (side, main), "side"], check=True)
edge = SCRATCH / "merged-shallow"
run(["git", "clone", "-q", "--depth", "1", "file://%s" % merged, edge], check=True)
at_edge = run(["python3", "-I", LINT, "--root", edge, "--range", "HEAD"])
print("merge at the edge:", at_edge.stdout.strip().splitlines()[-1])
assert at_edge.returncode == 0 and "commits 0, failed 0" in at_edge.stdout, at_edge.stdout
in_full = run(["python3", "-I", LINT, "--root", merged, "--range", "HEAD"])
assert in_full.returncode == 0 and "commits 3, failed 0" in in_full.stdout, in_full.stdout
bad_edge = SCRATCH / "bad-edge"
run(["git", "init", "-q", "-b", "main", bad_edge], check=True)
run(["git", *identity, "-C", bad_edge, "commit", "-q", "--allow-empty", "-m", message().rstrip("\n")], check=True)
run(["git", *identity, "-C", bad_edge, "commit", "-q", "--allow-empty", "-m", "essentials: fix it."], check=True)
bad_shallow = SCRATCH / "bad-shallow"
run(["git", "clone", "-q", "--depth", "1", "file://%s" % bad_edge, bad_shallow], check=True)
judged = run(["python3", "-I", LINT, "--root", bad_shallow, "--range", "HEAD"])
assert judged.returncode == 1 and "commits 1, failed 1" in judged.stdout, judged.stdout
print("a bad commit at the edge is still judged")

## The increment of cz bump

`.cz.toml` also tells `cz bump` which part of the version a commit raises. A breaking change,
written `type(scope)!: summary` or with a line that opens with `BREAKING CHANGE`, raises the major
version, `feat` the minor one, and `fix` and `perf` the patch. The cell tags version 1.0.0 in a
scratch repository, commits one message after the tag and asks `cz bump --dry-run` for the
increment.

In [ ]:
BUMP_CONFIG = CZ_CONFIG.read_text().replace("[tool.commitizen]\n", '[tool.commitizen]\nversion = "1.0.0"\n', 1)
assert 'version = "1.0.0"' in BUMP_CONFIG, "the configuration holds no table tool.commitizen"
INCREMENTS = [
    ("feat(api)!: drop the field uptime of Server", BODY, "MAJOR", "2.0.0"),
    ("feat(api): drop the field uptime of Server", BODY + "\n\nBREAKING CHANGE: the field uptime of Server is gone.",
     "MAJOR", "2.0.0"),
    ("feat(essentials): add the warp list", BODY, "MINOR", "1.1.0"),
    ("fix(essentials): refuse amounts below one cent in pay", BODY, "PATCH", "1.0.1"),
    ("perf(store): copy pointers instead of rows", BODY, "PATCH", "1.0.1"),
]
for number, (subject, body, expected, version) in enumerate(INCREMENTS):
    folder = SCRATCH / ("bump-%d" % number)
    run(["git", "init", "-q", "-b", "main", folder], check=True)
    (folder / "cz.toml").write_text(BUMP_CONFIG)
    run(["git", "-C", folder, "add", "cz.toml"], check=True)
    run(["git", *identity, "-C", folder, "commit", "-q", "-m", message(subject="chore(tests): start the history").rstrip("\n")],
        check=True)
    run(["git", "-C", folder, "tag", "1.0.0"], check=True)
    run(["git", *identity, "-C", folder, "commit", "-q", "--allow-empty", "-m",
         message(subject=subject, body=body).rstrip("\n")], check=True)
    bumped = run([sys.executable, "-m", "commitizen", "--config", folder / "cz.toml", "bump", "--dry-run", "--yes"],
                 cwd=folder)
    said = bumped.stdout + bumped.stderr
    print("%-55s %s" % (subject, " ".join(said.split())))
    assert bumped.returncode == 0 and "increment detected: %s" % expected in said, said
    assert "1.0.0 → %s" % version in said, said

## The questions of cz commit

`cz commit` asks for the parts of a message one question after the other. A body wrapped at 72
columns and the trailers take several lines, so their questions take several lines too, which
Enter separates and Esc and then Enter end. The cell answers the questions of `cz commit --dry-run`
on a terminal of its own and checks the message that Commitizen prints, which must keep every line
of the body and both trailers and pass the lint with the sign off that `-s` adds.

In [ ]:
import os
import pty
import re
import select
import subprocess
import time

TRAILERS = 'Closes: https://github.com/skytekx/golem/issues/12\nFixes: %s ("feat(essentials): add pay")' % FIXED
ANSWERS = [
    (b"The type of the change", b"\r"),
    (b"The scope", b"essentials\r"),
    (b"The summary", SUBJECT.split(": ", 1)[1].encode() + b"\r"),
    (b"The body", BODY.encode().replace(b"\n", b"\r") + b"\x1b\r"),
    (b"The trailers", TRAILERS.encode().replace(b"\n", b"\r") + b"\x1b\r"),
]
asked = SCRATCH / "asked"
run(["git", "init", "-q", "-b", "main", asked], check=True)
(asked / "a.txt").write_text("a\n")
run(["git", "-C", asked, "add", "a.txt"], check=True)
leader, follower = pty.openpty()
questions = subprocess.Popen(CZ + ["commit", "--dry-run"], cwd=asked, stdin=follower, stdout=follower,
                             stderr=follower, env=dict(os.environ, TERM="xterm"), close_fds=True)
os.close(follower)
screen = b""


def read_until(needle, seconds=60):
    # Reads what cz writes to its terminal until `needle` shows or cz ends, and answers the question
    # of prompt_toolkit for the position of the cursor.
    global screen
    deadline = time.monotonic() + seconds
    while needle is None or needle not in screen:
        assert time.monotonic() < deadline, "cz showed no %r:\n%s" % (needle, screen[-2000:])
        ready, _, _ = select.select([leader], [], [], 0.1)
        if not ready:
            continue
        try:
            chunk = os.read(leader, 1 << 16)
        except OSError:
            return
        if not chunk:
            return
        screen += chunk
        if b"\x1b[6n" in chunk:
            os.write(leader, b"\x1b[1;1R")


for question, answer in ANSWERS:
    read_until(question)
    os.write(leader, answer)
read_until(None)
assert questions.wait(timeout=60) == 0, screen[-2000:]
os.close(leader)
shown = re.sub(r"\x1b\[[0-9;?]*[A-Za-z]", "", screen.decode(errors="replace")).replace("\r", "")
written = shown[shown.rindex(SUBJECT + "\n\n"):].strip("\n")
print(written)
assert written == "%s\n\n%s\n\n%s" % (SUBJECT, BODY, TRAILERS), "cz wrote another message:\n" + written
(SCRATCH / "asked.txt").write_text(written + "\n" + sign_off + "\n")
signed = run(["python3", "-I", LINT, "--message-file", SCRATCH / "asked.txt", "--root", FIXTURE])
assert signed.returncode == 0, signed.stdout

## Verdict

The cell clears the scratch repository and prints the count. A run that reaches it passed every
case.

In [ ]:
shutil.rmtree(SCRATCH)
print("verdict: the commit lint passes the history and judges %d messages as expected, in agreement with Commitizen" % len(CASES))